# 4. Where the gain comes from: the time of one time step

A simulation advances the waves by one small time step at a time, thousands of times. At every step the
computer sends the GPU a few orders ("run this kernel on every grid point"), and the GPU reads and writes every
point of the grid. So the time of a step has two parts:

* a **fixed cost**: the time to send the orders and start the kernels, the same for a small and a large model;
* a **cost per grid point**: the time to read and write the memory of the GPU, which grows with the model.

This notebook times one step on the four engines for models from very small to large, then with several
sources, and counts the orders that OpenCL sends. It takes less than a minute.

In [ ]:
import os
import statistics
import sys

import matplotlib.pyplot as plt
import numpy as np

from comparison import (  # the code shared by the notebooks of this folder: comparison.py
    CUDA_FIXED,
    ENGINES,
    OPENCL_FIXED,
    OPENCL_LEGACY,
    adjoint_source,
    check_engines,
    colour,
    inclusion_model,
    line_style,
    name,
    propagator,
    seconds,
    style,
    survey,
)

check_engines()

## Settings

Square models of `SIZES` × `SIZES` grid points (7 m apart), one source, `T` seconds of recording (the number of
time steps does not depend on the size of the model, only on `T` and on the grid spacing). Each time is the median
of `REPEATS` runs after one run that is not timed.

In [ ]:
SIZES = [50, 100, 200, 300, 400, 600]
T = 0.3
REPEATS = 3

## The time of one step

A forward modelling without checkpoints (`chpr=0`): nothing but time steps. Its time divided by the number of
steps is the time of one step. The grid that PyFWI computes includes an absorbing border of 20 points on every
side (the PML), which the figure counts.

In [ ]:
per_step = {engine: [] for engine in ENGINES}  # milliseconds per time step, one per size
points = []  # grid points of each size, the border included
for size in SIZES:
    problem = survey(size, size, 1, T)
    model = inclusion_model(size, size)
    for engine in ENGINES:
        W = propagator(problem, engine, chpr=0)
        W.forward_modeling(model, show=False)  # compiles the kernels
        runs = [seconds(W.forward_modeling, model, show=False)[1] for _ in range(REPEATS)]
        per_step[engine].append(1e3 * statistics.median(runs) / W.nt)
    points.append(W.tnz * W.tnx)
    print(f"{size:4d} x {size:<4d} ({W.nt} steps) " + "   ".join(
        f"{name(engine)} {per_step[engine][-1]:.4f} ms" for engine in ENGINES))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))
style(left, grid_axis="both")
for engine in ENGINES:
    left.loglog(points, per_step[engine], line_style(engine), marker="o", color=colour(engine), label=name(engine))
left.set_xlabel("grid points (log scale)")
left.set_ylabel("milliseconds per time step (log scale)")
left.set_title("Time of one step of one source", loc="left", fontsize=10)
left.legend(frameon=False, fontsize=9)

style(right, grid_axis="both")
ratio = np.array(per_step[OPENCL_FIXED]) / np.array(per_step[CUDA_FIXED])
right.semilogx(points, ratio, marker="o", color=colour(CUDA_FIXED))
for x, value in zip(points, ratio):
    right.annotate(f"{value:.0f}x", (x, value), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=9)
right.set_ylim(0, ratio.max() * 1.2)
right.set_xlabel("grid points (log scale)")
right.set_ylabel("times faster on CUDA")
right.set_title("CUDA against OpenCL, both in the fixed mode", loc="left", fontsize=10)
fig.tight_layout()
plt.show()

Read the left figure from left to right:

* **On the smallest models the lines flatten**: making the model smaller hardly makes a step shorter any more.
  What is left is the fixed cost of a step: the GPU finishes its work before the next order arrives, and waits.
  CUDA in the `fixed` mode sits far below OpenCL there, because it does not send orders at every step: the steps
  between two checkpoints are recorded once as a *CUDA graph* and replayed by the GPU from a single order.
* **On larger models the lines rise with the number of grid points**: the work of the GPU itself is now the
  limit. The `fixed` CUDA kernels, written again for the GPU (the damping of the borders read only near the
  borders, single-precision numbers throughout), do that work several times faster than the kernels OpenCL runs.
* **The `legacy` lines of the two backends are close**: the same kernel files, sent step by step in both cases.

With one source, the right figure shows a gain of the same order at every size: the two causes, fewer orders and
faster kernels, weigh about the same here.

## Several sources at once

A survey has many sources. OpenCL simulates them one after the other. CUDA runs several in the same kernels: every
launch covers every source of a batch, so the fixed cost of a step is shared between them. Here the time of one
step divided by the number of sources, on a 100 × 100 model:

In [ ]:
SOURCES = [1, 2, 4, 8]
size = SIZES[1]
per_source = {engine: [] for engine in (OPENCL_FIXED, CUDA_FIXED)}  # milliseconds per step and per source
for ns in SOURCES:
    problem = survey(size, size, ns, T)
    model = inclusion_model(size, size)
    for engine, times in per_source.items():
        W = propagator(problem, engine, chpr=0)
        W.forward_modeling(model, show=False)
        runs = [seconds(W.forward_modeling, model, show=False)[1] for _ in range(REPEATS)]
        times.append(1e3 * statistics.median(runs) / (W.nt * ns))
    print(f"{ns} source{'s' if ns > 1 else ' '}: " + "   ".join(f"{name(engine)} {per_source[engine][-1]:.4f} ms" for engine in per_source)
          + f"   CUDA {per_source[OPENCL_FIXED][-1] / per_source[CUDA_FIXED][-1]:.1f} times faster")

fig, ax = plt.subplots(figsize=(7, 4))
style(ax)
width = 0.38
for e, engine in enumerate(per_source):
    x = np.arange(len(SOURCES)) + (e - 0.5) * width
    ax.bar(x, per_source[engine], width, color=colour(engine), label=name(engine))
    for xi, value in zip(x, per_source[engine]):
        ax.text(xi, value, f"{value:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_ylim(0, max(max(values) for values in per_source.values()) * 1.3)
ax.set_xticks(np.arange(len(SOURCES)), [f"{ns} source{'s' if ns > 1 else ''}" for ns in SOURCES])
ax.set_ylabel("milliseconds per step and per source")
ax.set_title(f"{size} x {size} model: the time of one step, shared between the sources", loc="left", fontsize=10)
ax.legend(frameon=False, ncol=2, loc="upper left")
plt.show()

On OpenCL the time per source does not change: each source costs a full step. On CUDA it falls as the sources
share the fixed cost, until the GPU is full of work: beyond that, more sources at once no longer help (on the
RTX 3060 of the project, 8 sources took longer per source than 4). That is why
[notebook 1](1_speed.ipynb) found the largest gain on the small model with 4 sources: there OpenCL pays its fixed
cost four times per step and CUDA once.

## Counting the orders of OpenCL

`tools/benchmark.py`, the benchmark of the repository, counts the commands that PyFWI sends through PyOpenCL:
kernel launches and copies between the memory of the computer and that of the GPU. Here, per time step of one
source, for a forward modelling and a gradient with checkpoints:

In [ ]:
sys.path.insert(0, os.path.abspath(os.path.join("..", "..")))  # the root of the repository, for tools/
from tools.benchmark import count_opencl_calls

problem = survey(SIZES[1], SIZES[1], 1, T)
true, start = inclusion_model(SIZES[1], SIZES[1]), inclusion_model(SIZES[1], SIZES[1], smooth=1)
for engine in (OPENCL_LEGACY, OPENCL_FIXED):
    W = propagator(problem, engine, chpr=20)
    observed = W.forward_modeling(true, show=False)
    with count_opencl_calls() as forward:
        predicted = W.forward_modeling(start, show=False)
    _, source = adjoint_source(W, predicted, observed)
    with count_opencl_calls() as gradient:
        W.gradient(source, show=False)
    print(f"{name(engine):15s} forward modelling: {forward['launches'] / W.nt:.1f} kernels and "
          f"{forward['copies'] / W.nt:.1f} copies per step;   gradient: {gradient['launches'] / W.nt:.1f} kernels "
          f"and {gradient['copies'] / W.nt:.1f} copies per step")

Every one of these commands costs the computer some microseconds, thousands of times per simulation. The copies
are the checkpoints: OpenCL copies the wave field to the memory of the computer as the forward modelling goes, and
back during the gradient. CUDA keeps the checkpoints in the memory of the GPU when they fit
([notebook 5](5_memory.ipynb)), and replays its steps from graphs, so there is nothing comparable to count. The
[final document](../../docs/cuda-migration/cuda-vs-opencl.md) measures the share of the time the GPU works on each
backend, with a timeline of the GPU.